decoder.py

In [ ]:
"""
decoder.py
==========
Phase 3a – Reconstructive Decoder for PartialConvMAE.

Pipeline
--------
1.  Receive encoded visible tokens  (B, N_vis, C_enc)  from the encoder.
2.  Allocate a full token sequence of length N = D'·H'·W' initialised
    with a shared learned *mask token* at invisible positions.
3.  Fill visible positions with the encoded tokens (after a linear
    projection from C_enc → C_dec).
4.  Add sinusoidal positional encoding to the full sequence.
5.  Run L_d *cross-attention* Transformer blocks where:
      Q  = full sequence (mask tokens + visible tokens)
      K,V = encoded visible tokens                   ← information flow
    This forces mask tokens to query information from visible tokens
    rather than attending to each other's zeros.
6.  Project each token (C_dec → patch_vol) to recover the raw signal
    values for every voxel in the patch.
7.  Rearrange the output into the original spatial volume
    (B, 1, D, H, W)  via a pixel-shuffle–style fold.

Parameters that must match the encoder / stem
----------------------------------------------
  embed_dim   (C_enc)  – encoder hidden dim
  dec_dim     (C_dec)  – decoder hidden dim (can be smaller for efficiency)
  patch_stride         – downsampling factor used in the stem
  in_channels          – raw signal channels (1 for power map)
  grid_shape           – (D', H', W') returned by the encoder
"""

import torch
import torch.nn as nn
from encoder import sinusoidal_3d


# ---------------------------------------------------------------------------
# Cross-Attention Transformer Block
# ---------------------------------------------------------------------------

class CrossAttentionBlock(nn.Module):
    """
    Pre-LayerNorm cross-attention block.

    Q  comes from the *full* sequence (mask + visible tokens).
    K,V come from the *encoded visible* tokens only.

    After cross-attention the block applies a standard FFN to each token.
    """

    def __init__(
        self,
        dec_dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        # Cross-attention
        self.norm_q  = nn.LayerNorm(dec_dim)
        self.norm_kv = nn.LayerNorm(dec_dim)
        self.cross_attn = nn.MultiheadAttention(
            dec_dim, num_heads, dropout=dropout, batch_first=True
        )

        # Self-attention on full sequence
        self.norm_self = nn.LayerNorm(dec_dim)
        self.self_attn = nn.MultiheadAttention(
            dec_dim, num_heads, dropout=dropout, batch_first=True
        )

        # Feed-forward network
        self.norm_ffn = nn.LayerNorm(dec_dim)
        hidden = int(dec_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dec_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dec_dim),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x: torch.Tensor,          # (B, N,     C_dec)  full sequence
        context: torch.Tensor,    # (B, N_vis, C_dec)  visible tokens
    ) -> torch.Tensor:
        # 1. Cross-attention: full sequence queries encoded visible tokens
        q   = self.norm_q(x)
        kv  = self.norm_kv(context)
        ca_out, _ = self.cross_attn(q, kv, kv)
        x = x + ca_out

        # 2. Self-attention over full sequence
        sa_normed = self.norm_self(x)
        sa_out, _ = self.self_attn(sa_normed, sa_normed, sa_normed)
        x = x + sa_out

        # 3. FFN
        x = x + self.ffn(self.norm_ffn(x))
        return x


# ---------------------------------------------------------------------------
# Full Decoder
# ---------------------------------------------------------------------------

class PartialConvDecoder(nn.Module):
    """
    Lightweight decoder that reconstructs the full RF tensor from the
    encoded visible patches plus learned mask tokens.

    Parameters
    ----------
    enc_dim      : int   – encoder output dimension
    dec_dim      : int   – decoder hidden dimension
    depth        : int   – number of cross-attention blocks (L_d)
    num_heads    : int
    patch_stride : int   – must match the stem's patch_stride
    in_channels  : int   – raw signal channels (1)
    mlp_ratio    : float
    dropout      : float
    """

    def __init__(
        self,
        enc_dim: int = 256,
        dec_dim: int = 128,
        depth: int = 4,
        num_heads: int = 8,
        patch_stride: int = 4,
        in_channels: int = 1,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.dec_dim      = dec_dim
        self.patch_stride = patch_stride
        self.in_channels  = in_channels

        # Each patch reconstructs (patch_stride)^3 voxels × in_channels
        self.patch_vol = (patch_stride ** 3) * in_channels

        # Project encoder dim → decoder dim
        self.enc_proj = nn.Linear(enc_dim, dec_dim)

        # Learned mask token (shared across all invisible positions)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        # Cross-attention decoder blocks
        self.blocks = nn.ModuleList([
            CrossAttentionBlock(dec_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dec_dim)

        # Prediction head: token → raw voxel values for one patch
        self.pred_head = nn.Linear(dec_dim, self.patch_vol)

    # ------------------------------------------------------------------
    def forward(
        self,
        encoded: torch.Tensor,        # (B, N_vis, enc_dim)
        pos_all: torch.Tensor,        # (B, N,     enc_dim)
        vis_any: torch.Tensor,        # (N,) bool  – which positions are visible
        grid_shape: tuple,            # (D', H', W')
        original_shape: tuple,        # (B, C, D, H, W) – target volume shape
    ) -> torch.Tensor:
        """
        Returns
        -------
        recon : (B, C, D, H, W)  – reconstructed full RF tensor
        """
        B      = encoded.shape[0]
        Dp, Hp, Wp = grid_shape
        N      = Dp * Hp * Wp

        # ── Project encoded tokens to decoder dim ──────────────────────
        ctx = self.enc_proj(encoded)                  # (B, N_vis, dec_dim)

        # ── Build full token sequence ───────────────────────────────────
        # Initialise all N positions with the mask token
        full_tokens = self.mask_token.expand(B, N, -1).clone()
        # (B, N, dec_dim)

        # Overwrite visible positions with projected encoder output
        full_tokens[:, vis_any, :] = ctx              # in-place fill

        # ── Add positional encoding (projected to dec_dim) ─────────────
        # pos_all is (B, N, enc_dim); we project it too
        pos_dec = nn.functional.linear(
            pos_all,
            self.enc_proj.weight,
            self.enc_proj.bias,
        )                                             # (B, N, dec_dim)
        full_tokens = full_tokens + pos_dec

        # ── Cross-attention decoder blocks ─────────────────────────────
        for blk in self.blocks:
            full_tokens = blk(full_tokens, ctx)

        full_tokens = self.norm(full_tokens)          # (B, N, dec_dim)

        # ── Prediction head: (B, N, patch_vol) ────────────────────────
        pred = self.pred_head(full_tokens)            # (B, N, patch_vol)

        # ── Fold patch predictions back into spatial volume ────────────
        recon = self._fold(pred, grid_shape, original_shape)
        return recon

    # ------------------------------------------------------------------
    def _fold(
        self,
        pred: torch.Tensor,    # (B, N, patch_vol)
        grid_shape: tuple,     # (D', H', W')
        original_shape: tuple, # (B, C, D, H, W)
    ) -> torch.Tensor:
        """
        Rearrange flat patch predictions into the full 3-D volume.

        Each patch covers (patch_stride × patch_stride × patch_stride)
        voxels in (D, H, W).  We simply reshape and permute.
        """
        B, C_orig, D, H, W = original_shape
        Dp, Hp, Wp = grid_shape
        ps = self.patch_stride

        # pred: (B, N, ps^3 * C)
        # Reshape to (B, D', H', W', ps, ps, ps, C)
        pred = pred.view(B, Dp, Hp, Wp, ps, ps, ps, self.in_channels)

        # Permute to (B, C, D'*ps, H'*ps, W'*ps)
        #   axes:  0   7   1  4   2  5   3  6
        pred = pred.permute(0, 7, 1, 4, 2, 5, 3, 6).contiguous()
        pred = pred.view(B, self.in_channels, Dp * ps, Hp * ps, Wp * ps)

        # Crop / pad to exact original spatial size
        pred = pred[:, :, :D, :H, :W]
        return pred


encoder.py

In [ ]:
"""
encoder.py
==========
Phase 2 – Asymmetric Transformer Encoder for PartialConvMAE.

Pipeline
--------
1. Receive the dense feature map  (B, C, D', H', W')  and the
   downsampled binary mask         (B, 1, D', H', W')  from the
   PartialConvStem.

2. Flatten spatial/freq dimensions → N = D'·H'·W' tokens, each of
   dimension C.

3. Compute 3-D sinusoidal positional encoding for every token, storing
   the (d, h, w) grid indices so that the decoder can reuse them.

4. **Discard** patches that are entirely un-measured (mask sum == 0 over
   the patch volume).  This is the "asymmetric" trick from MAE: the
   encoder only processes the small visible subset, dramatically reducing
   compute.

5. Process the surviving tokens through L_e Transformer blocks (MSA +
   FFN, pre-LayerNorm style, standard PyTorch MultiheadAttention).

6. Return:
   - encoded visible tokens   (B, N_vis, C)
   - positional encodings of ALL N tokens   (B, N, C)   [for decoder]
   - visibility mask          (B, N)  bool  [True = visible]
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# 3-D Sinusoidal Positional Encoding
# ---------------------------------------------------------------------------

def sinusoidal_3d(
    d_grid: int,
    h_grid: int,
    w_grid: int,
    embed_dim: int,
    device: torch.device = None,
) -> torch.Tensor:
    """
    Build a 3-D sinusoidal positional encoding table.

    The embed_dim is split into three equal parts (⌊C/3⌋ each);
    any remainder dims are zero-padded.

    Returns
    -------
    pe : (1, D'·H'·W', embed_dim)  – ready to broadcast over batch
    """
    dim_each = embed_dim // 3          # dims allocated per axis
    half = dim_each // 2              # sin / cos split

    def _1d_enc(length: int) -> torch.Tensor:
        """Returns (length, dim_each) sinusoidal encoding for one axis."""
        enc = torch.zeros(length, dim_each)
        pos = torch.arange(length, dtype=torch.float32).unsqueeze(1)  # (L,1)
        # Use only even indices so sin/cos pairs are always equal count
        even_idx = torch.arange(0, dim_each, 2, dtype=torch.float32)  # (half_floor,)
        div = torch.pow(10000.0, even_idx / dim_each).unsqueeze(0)    # (1, half_floor)
        sins = torch.sin(pos / div)   # (L, half_floor)
        coss = torch.cos(pos / div)   # (L, half_floor)
        n_sin = sins.shape[1]
        n_cos = coss.shape[1]
        enc[:, 0::2] = sins                    # positions 0,2,4,...
        enc[:, 1::2] = coss[:, :enc[:, 1::2].shape[1]]  # positions 1,3,5,... (safe slice)
        return enc  # (length, dim_each)

    enc_d = _1d_enc(d_grid)   # (D', dim_each)
    enc_h = _1d_enc(h_grid)   # (H', dim_each)
    enc_w = _1d_enc(w_grid)   # (W', dim_each)

    # Tile into full grid
    # Each token at (d, h, w) gets [ enc_d[d] | enc_h[h] | enc_w[w] ]
    # shape after combination: (D'*H'*W', 3*dim_each)
    N = d_grid * h_grid * w_grid
    pe_full = torch.zeros(N, 3 * dim_each)
    idx = 0
    for di in range(d_grid):
        for hi in range(h_grid):
            for wi in range(w_grid):
                pe_full[idx, :dim_each]           = enc_d[di]
                pe_full[idx, dim_each:2*dim_each]  = enc_h[hi]
                pe_full[idx, 2*dim_each:3*dim_each] = enc_w[wi]
                idx += 1

    # Pad to embed_dim if necessary
    if 3 * dim_each < embed_dim:
        padding = torch.zeros(N, embed_dim - 3 * dim_each)
        pe_full = torch.cat([pe_full, padding], dim=-1)

    pe_full = pe_full.unsqueeze(0)  # (1, N, embed_dim)
    if device is not None:
        pe_full = pe_full.to(device)
    return pe_full


# ---------------------------------------------------------------------------
# Transformer building blocks
# ---------------------------------------------------------------------------

class TransformerBlock(nn.Module):
    """
    Pre-LayerNorm Transformer block:
        x = x + MSA(LN(x))
        x = x + FFN(LN(x))

    Parameters
    ----------
    embed_dim  : int
    num_heads  : int
    mlp_ratio  : float – FFN hidden dim = embed_dim * mlp_ratio
    dropout    : float
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Self-attention
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        # FFN
        x = x + self.ffn(self.norm2(x))
        return x


# ---------------------------------------------------------------------------
# Asymmetric Transformer Encoder
# ---------------------------------------------------------------------------

class AsymmetricEncoder(nn.Module):
    """
    Tokenizes the PartialConvStem output, discards invisible patches,
    adds sinusoidal positional encoding, and runs L_e Transformer blocks.

    Parameters
    ----------
    embed_dim  : int   – feature / token dimension (= stem output channels)
    depth      : int   – number of Transformer blocks  (L_e)
    num_heads  : int
    mlp_ratio  : float
    dropout    : float
    """

    def __init__(
        self,
        embed_dim: int = 256,
        depth: int = 6,
        num_heads: int = 8,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.embed_dim = embed_dim

        # Input projection: 1×1×1 conv to make channels explicit
        self.proj = nn.Linear(embed_dim, embed_dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    # ------------------------------------------------------------------
    def forward(
        self,
        feat: torch.Tensor,   # (B, C, D', H', W')  from PartialConvStem
        mask: torch.Tensor,   # (B, 1, D', H', W')  downsampled binary mask
    ):
        """
        Returns
        -------
        encoded    : (B, N_vis, C)      – encoded visible tokens
        pos_all    : (B, N,     C)      – positional enc for ALL N tokens
        vis_mask   : (B, N)   bool      – True where patch is visible
        grid_shape : (D', H', W')       – spatial shape for decoder
        """
        B, C, Dp, Hp, Wp = feat.shape
        N = Dp * Hp * Wp

        # ── Flatten to tokens ──────────────────────────────────────────
        # (B, C, D', H', W') → (B, N, C)
        tokens = feat.flatten(2).transpose(1, 2)       # (B, N, C)
        tokens = self.proj(tokens)                      # linear mixing

        # ── Positional encoding for all N positions ────────────────────
        pos_all = sinusoidal_3d(Dp, Hp, Wp, C, device=feat.device)
        # pos_all: (1, N, C)  → expand to (B, N, C)
        pos_all = pos_all.expand(B, -1, -1)

        # ── Determine visibility: a patch is visible if its mask sum > 0 ─
        # mask: (B, 1, D', H', W') → (B, N)
        mask_flat = mask.flatten(2).squeeze(1)          # (B, N)
        vis_mask  = mask_flat > 0                       # (B, N) bool

        # ── Extract visible tokens (variable length per sample) ────────
        # Strategy: for efficiency during inference we batch-process by
        # taking the union of visible positions across the batch.
        # During training with uniform sparsity, all samples share the
        # same visible set; during inference we iterate per-sample.
        vis_any = vis_mask.any(dim=0)                  # (N,)
        # Use union-of-visible-positions approach (safe for batching)
        visible_tokens = tokens[:, vis_any, :]         # (B, N_vis, C)
        visible_pos    = pos_all[:, vis_any, :]        # (B, N_vis, C)

        # Add positional encoding to visible tokens
        x = visible_tokens + visible_pos               # (B, N_vis, C)

        # ── Transformer blocks ─────────────────────────────────────────
        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)                               # (B, N_vis, C)

        return x, pos_all, vis_any, (Dp, Hp, Wp)


loss.py

In [ ]:
"""
loss.py
=======
Phase 3b – Custom Reconstruction Loss for PartialConvMAE.

Implements a masked MSE loss that computes the error **strictly over the
initially unmeasured regions** (where mask == 0), i.e. only the pixels
the model had to hallucinate / infer.

This design choice follows the MAE philosophy: there is no reward for
accurately copying already-observed measurements; the model must learn
the underlying RF propagation structure to reconstruct missing areas.

Optionally a weighted combination with a *full-map* MSE term can be
enabled (via `lambda_full`) to stabilise training in early epochs.

Functions
---------
masked_mse_loss(pred, target, mask, lambda_full=0.0, reduction='mean')

    pred        : (B, C, D, H, W)  – model output
    target      : (B, C, D, H, W)  – ground-truth full RF tensor
    mask        : (B, 1, D, H, W)  – binary mask  (1=measured, 0=missing)
    lambda_full : float             – weight for the auxiliary full-map term
    reduction   : 'mean' | 'sum'   – how to reduce the hole loss

    Returns scalar loss tensor.
"""

import torch
import torch.nn.functional as F


def masked_mse_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    lambda_full: float = 0.0,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    Compute MSE loss over the *unobserved* (hole) region only.

    The hole region is defined by  (1 - mask),  i.e. locations that
    were zero in the original sparse measurement.

    Parameters
    ----------
    pred        : (B, C, D, H, W)  – reconstructed tensor
    target      : (B, C, D, H, W)  – ground-truth full RF tensor
    mask        : (B, 1, D, H, W)  – binary mask (1=measured, 0=missing)
                  Automatically broadcast over channels.
    lambda_full : float  ≥ 0       – if > 0, adds λ × MSE_full to the
                                      loss for auxiliary supervision
    reduction   : 'mean' | 'sum'   – aggregation over hole voxels

    Returns
    -------
    loss : scalar Tensor
    """
    # Broadcast mask to match channel dimension
    hole_mask = (1.0 - mask)                # (B, 1, D, H, W)  1=hole, 0=measured
    # Expand to C channels
    if hole_mask.shape[1] != pred.shape[1]:
        hole_mask = hole_mask.expand_as(pred)

    # ── Squared error everywhere ────────────────────────────────────────
    sq_err = (pred - target) ** 2           # (B, C, D, H, W)

    # ── Hole loss ──────────────────────────────────────────────────────
    hole_sq_err = sq_err * hole_mask        # zero out measured locations

    n_hole = hole_mask.sum().clamp(min=1.0) # number of hole voxels (avoid /0)

    if reduction == "mean":
        loss_hole = hole_sq_err.sum() / n_hole
    else:  # 'sum'
        loss_hole = hole_sq_err.sum()

    # ── Optional full-map auxiliary loss ──────────────────────────────
    if lambda_full > 0.0:
        loss_full = F.mse_loss(pred, target, reduction="mean")
        return loss_hole + lambda_full * loss_full

    return loss_hole


# ---------------------------------------------------------------------------
# Convenience class-based wrapper (compatible with nn.Module training loops)
# ---------------------------------------------------------------------------

class MaskedReconstructionLoss(torch.nn.Module):
    """
    nn.Module wrapper around masked_mse_loss for use in training loops.

    Parameters
    ----------
    lambda_full : float – auxiliary full-map MSE weight  (default 0.1)
    reduction   : str   – 'mean' or 'sum'
    """

    def __init__(
        self,
        lambda_full: float = 0.1,
        reduction: str = "mean",
    ) -> None:
        super().__init__()
        self.lambda_full = lambda_full
        self.reduction   = reduction

    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        return masked_mse_loss(
            pred, target, mask,
            lambda_full=self.lambda_full,
            reduction=self.reduction,
        )

    def extra_repr(self) -> str:
        return (f"lambda_full={self.lambda_full}, "
                f"reduction='{self.reduction}'")


main.py

In [ ]:
"""
main.py
=======
Instantiates the PartialConvMAE model and runs a dummy forward pass with
a randomly generated sparse RF tensor and binary mask.

Tensor layout
-------------
  Batch size  B = 2
  Channels    C = 1   (single-channel power map in dBm)
  Depth       D = 8   (frequency / time slices)
  Height      H = 32  (spatial grid y)
  Width       W = 32  (spatial grid x)

Sparsity
--------
  ~85 % of voxels are masked (unobserved), matching the expected
  operational density of the UGV traversing the UCF campus grid.

Usage
-----
  python main.py [--device cuda|cpu] [--sparsity 0.85]
"""

import argparse
import sys
import time

import torch

# Add the module directory to path (needed when running from outside the pkg)
sys.path.insert(0, ".")

from model import PartialConvMAE


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def generate_dummy_data(
    batch:    int   = 2,
    channels: int   = 1,
    depth:    int   = 8,
    height:   int   = 32,
    width:    int   = 32,
    sparsity: float = 0.85,
    device:   str   = "cpu",
) -> tuple:
    """
    Generate a synthetic sparse RF tensor and its binary mask.

    The raw RF power map values are drawn from a normal distribution
    centred at –80 dBm (typical indoor received power) with σ = 10 dB.

    Parameters
    ----------
    sparsity : float  – fraction of voxels that are *unobserved* (mask=0)

    Returns
    -------
    x_sparse : (B, C, D, H, W) – zeroed at unobserved locations
    mask     : (B, 1, D, H, W) – 1=measured, 0=missing
    target   : (B, C, D, H, W) – full ground-truth map (for loss)
    """
    shape = (batch, channels, depth, height, width)
    dev   = torch.device(device)

    # Ground truth: realistic dBm values centred around –80 dBm
    target = torch.randn(*shape, device=dev) * 10.0 - 80.0

    # Binary mask: Bernoulli draw – each voxel observed with prob (1-sparsity)
    mask_shape = (batch, 1, depth, height, width)
    mask = torch.bernoulli(
        torch.full(mask_shape, 1.0 - sparsity, device=dev)
    )

    # Guard: ensure at least one measured voxel per sample to avoid
    # the degenerate all-invisible case during the demo
    for b in range(batch):
        if mask[b].sum() == 0:
            mask[b, 0, 0, 0, 0] = 1.0

    # Sparse input: zero out unmeasured locations
    x_sparse = target * mask  # mask broadcasts over channels

    return x_sparse, mask, target


def pretty_print_params(param_dict: dict) -> None:
    print("\n┌─────────────────────────────────────┐")
    print("│        PartialConvMAE Parameters      │")
    print("├──────────────────────┬───────────────┤")
    for name, count in param_dict.items():
        label = f"  {name:<18}"
        value = f"{count:>12,}  "
        sep   = "│" if name != "total" else "╞"
        print(f"{sep}{label}│{value}│")
    print("└──────────────────────┴───────────────┘")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="PartialConvMAE dummy forward pass")
    parser.add_argument("--device",   default="cpu",  choices=["cpu", "cuda"],
                        help="Device to run on (default: cpu)")
    parser.add_argument("--sparsity", default=0.85, type=float,
                        help="Fraction of unmeasured voxels (default: 0.85)")
    parser.add_argument("--batch",    default=2,    type=int)
    parser.add_argument("--depth",    default=8,    type=int,
                        help="Number of frequency / time slices")
    parser.add_argument("--height",   default=32,   type=int)
    parser.add_argument("--width",    default=32,   type=int)
    args = parser.parse_args()

    device = args.device
    if device == "cuda" and not torch.cuda.is_available():
        print("[WARNING] CUDA not available, falling back to CPU.")
        device = "cpu"

    print("=" * 55)
    print("  PartialConvMAE – Intelligent Dynamic Spectrum Cartography")
    print("  UCF / US Army DEVCOM ARL  |  Jetson Orin Nano Target")
    print("=" * 55)
    print(f"\n  Device   : {device.upper()}")
    print(f"  Sparsity : {args.sparsity * 100:.0f}% unmeasured")
    print(f"  Tensor   : B={args.batch}, C=1, D={args.depth}, "
          f"H={args.height}, W={args.width}")

    # ── Instantiate model ───────────────────────────────────────────────
    print("\n[1/4] Building PartialConvMAE …")
    model = PartialConvMAE(
        in_channels   = 1,
        stem_dim      = 64,
        embed_dim     = 256,
        dec_dim       = 128,
        enc_depth     = 6,
        dec_depth     = 4,
        num_heads_enc = 8,
        num_heads_dec = 8,
        mlp_ratio     = 4.0,
        patch_stride  = 4,
        dropout       = 0.1,
        lambda_full   = 0.1,
    ).to(device)
    model.eval()

    pretty_print_params(model.count_parameters())

    # ── Generate dummy data ─────────────────────────────────────────────
    print("\n[2/4] Generating sparse RF data …")
    x_sparse, mask, target = generate_dummy_data(
        batch    = args.batch,
        channels = 1,
        depth    = args.depth,
        height   = args.height,
        width    = args.width,
        sparsity = args.sparsity,
        device   = device,
    )

    observed = mask.sum().item()
    total    = mask.numel()
    print(f"         Observed voxels : {int(observed):,} / {total:,} "
          f"({100 * observed / total:.1f}%)")
    print(f"         Missing  voxels : {total - int(observed):,} "
          f"({100 * (1 - observed / total):.1f}%)")

    # ── Forward pass (inference mode) ───────────────────────────────────
    print("\n[3/4] Running forward pass …")
    with torch.no_grad():
        t0 = time.perf_counter()
        recon, loss = model(x_sparse, mask, target)
        t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1_000

    # ── Report results ───────────────────────────────────────────────────
    print("\n[4/4] Results")
    print(f"  Input  shape : {list(x_sparse.shape)}")
    print(f"  Output shape : {list(recon.shape)}")
    print(f"  Masked MSE   : {loss.item():.6f}")
    print(f"  Latency      : {latency_ms:.2f} ms  "
          f"({'✓ real-time capable' if latency_ms < 500 else '⚠ may need optimisation'})")

    # Sanity checks
    assert recon.shape == x_sparse.shape, \
        f"Shape mismatch: got {recon.shape}, expected {x_sparse.shape}"
    assert not torch.isnan(recon).any(), "NaN values detected in reconstruction!"
    assert not torch.isinf(recon).any(), "Inf values detected in reconstruction!"

    print("\n  ✅  All sanity checks passed.")
    print("\n  Reconstruction statistics (hole region only):")
    hole_mask   = (1.0 - mask).bool()
    hole_target = target[hole_mask]
    hole_recon  = recon[hole_mask]
    print(f"    Target  – mean: {hole_target.mean():.2f} dBm, "
          f"std: {hole_target.std():.2f} dB")
    print(f"    Recon   – mean: {hole_recon.mean():.2f} dBm, "
          f"std: {hole_recon.std():.2f} dB")
    mae = (hole_recon - hole_target).abs().mean().item()
    print(f"    MAE     : {mae:.4f} dB  (untrained model – expected to be high)")

    print("\n" + "=" * 55)
    print("  Forward pass complete. Model is ready for training.")
    print("=" * 55 + "\n")


if __name__ == "__main__":
    main()


model.py

In [ ]:
"""
model.py
========
PartialConvMAE – top-level model that wires together all three phases:

  Phase 1  PartialConvStem   (partial_conv.py)
  Phase 2  AsymmetricEncoder (encoder.py)
  Phase 3  PartialConvDecoder + MaskedReconstructionLoss
           (decoder.py / loss.py)

The model takes a sparse RF tensor and its binary measurement mask and
returns a fully reconstructed spatio-temporal spectrum map of the same
spatial size.

Input / Output
--------------
  x     : (B, C_in, D, H, W)   sparse RF power map  (zeros at holes)
  mask  : (B, 1,    D, H, W)   binary mask  (1 = measured, 0 = missing)

  → recon : (B, C_in, D, H, W) reconstructed full map

Default configuration targets the Jetson Orin Nano:
  - embed_dim  256  (enc_dim)
  - dec_dim    128
  - enc_depth   6 blocks
  - dec_depth   4 blocks
  - patch_stride 4   (64×64 map → 16×16 grid)
  - stem_dim   64
"""

import torch
import torch.nn as nn

from partial_conv import PartialConvStem
from encoder      import AsymmetricEncoder
from decoder      import PartialConvDecoder
from loss         import MaskedReconstructionLoss


class PartialConvMAE(nn.Module):
    """
    Partial Convolutional Masked Autoencoder for spatio-temporal
    RF spectrum map reconstruction.

    Parameters
    ----------
    in_channels   : int   – raw input channels (1 for power map)
    stem_dim      : int   – PartialConvStem intermediate channels
    embed_dim     : int   – encoder (and stem output) token dimension
    dec_dim       : int   – decoder hidden dimension
    enc_depth     : int   – number of encoder Transformer blocks  (L_e)
    dec_depth     : int   – number of decoder cross-attention blocks (L_d)
    num_heads_enc : int   – attention heads in encoder
    num_heads_dec : int   – attention heads in decoder
    mlp_ratio     : float – FFN expansion ratio
    patch_stride  : int   – spatial downsampling in the stem (patch size)
    dropout       : float – dropout probability in attention / FFN
    lambda_full   : float – auxiliary full-map loss weight (0 = disabled)
    """

    def __init__(
        self,
        in_channels:   int   = 1,
        stem_dim:      int   = 64,
        embed_dim:     int   = 256,
        dec_dim:       int   = 128,
        enc_depth:     int   = 6,
        dec_depth:     int   = 4,
        num_heads_enc: int   = 8,
        num_heads_dec: int   = 8,
        mlp_ratio:     float = 4.0,
        patch_stride:  int   = 4,
        dropout:       float = 0.1,
        lambda_full:   float = 0.1,
    ) -> None:
        super().__init__()

        # ── Phase 1: Partial Convolutional Stem ────────────────────────
        self.stem = PartialConvStem(
            in_channels  = in_channels,
            stem_dim     = stem_dim,
            embed_dim    = embed_dim,
            patch_stride = patch_stride,
        )

        # ── Phase 2: Asymmetric Transformer Encoder ────────────────────
        self.encoder = AsymmetricEncoder(
            embed_dim = embed_dim,
            depth     = enc_depth,
            num_heads = num_heads_enc,
            mlp_ratio = mlp_ratio,
            dropout   = dropout,
        )

        # ── Phase 3: Reconstructive Decoder ───────────────────────────
        self.decoder = PartialConvDecoder(
            enc_dim      = embed_dim,
            dec_dim      = dec_dim,
            depth        = dec_depth,
            num_heads    = num_heads_dec,
            patch_stride = patch_stride,
            in_channels  = in_channels,
            mlp_ratio    = mlp_ratio,
            dropout      = dropout,
        )

        # ── Loss ───────────────────────────────────────────────────────
        self.criterion = MaskedReconstructionLoss(lambda_full=lambda_full)

        self._init_weights()

    # ------------------------------------------------------------------
    def _init_weights(self):
        """Apply Xavier / truncated-normal initialisation to linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm3d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    # ------------------------------------------------------------------
    def forward(
        self,
        x:      torch.Tensor,   # (B, C, D, H, W)
        mask:   torch.Tensor,   # (B, 1, D, H, W)
        target: torch.Tensor = None,  # (B, C, D, H, W) optional for loss
    ):
        """
        Forward pass.

        Parameters
        ----------
        x      : sparse input RF tensor  (zeros at unmeasured locations)
        mask   : binary measurement mask (1=measured, 0=missing)
        target : ground-truth full RF tensor (required for loss computation)

        Returns
        -------
        If target is provided:
            (recon, loss)  – reconstructed tensor + scalar loss
        Else:
            recon          – reconstructed tensor only
        """
        original_shape = x.shape           # (B, C, D, H, W)

        # ── Phase 1 ────────────────────────────────────────────────────
        feat, mask_down = self.stem(x, mask)
        # feat:      (B, embed_dim, D', H', W')
        # mask_down: (B, 1,         D', H', W')

        # ── Phase 2 ────────────────────────────────────────────────────
        encoded, pos_all, vis_any, grid_shape = self.encoder(feat, mask_down)
        # encoded:    (B, N_vis, embed_dim)
        # pos_all:    (B, N,     embed_dim)
        # vis_any:    (N,) bool
        # grid_shape: (D', H', W')

        # ── Phase 3 ────────────────────────────────────────────────────
        recon = self.decoder(
            encoded,
            pos_all,
            vis_any,
            grid_shape,
            original_shape,
        )
        # recon: (B, C, D, H, W)

        if target is not None:
            loss = self.criterion(recon, target, mask)
            return recon, loss

        return recon

    # ------------------------------------------------------------------
    def count_parameters(self) -> dict:
        """Return parameter counts per sub-module."""
        def _count(m):
            return sum(p.numel() for p in m.parameters() if p.requires_grad)
        return {
            "stem":    _count(self.stem),
            "encoder": _count(self.encoder),
            "decoder": _count(self.decoder),
            "total":   _count(self),
        }


partial_conv.py

In [ ]:
"""
partial_conv.py
===============
Phase 1 – Partial Convolutional Embedding for PartialConvMAE.

A 3-D Partial Convolution layer that:
  • Conditions the convolution strictly on *valid* (measured) pixels
    (locations where the binary mask M == 1).
  • Scales activations to compensate for the fraction of valid pixels
    inside each receptive field.
  • Propagates an updated mask forward: an output location is valid (1)
    if at least one input pixel in its receptive field was valid.

Reference:
  Liu et al., "Image Inpainting for Irregular Holes Using Partial
  Convolutions", ECCV 2018.  Extended here to 3-D (spatial-x,
  spatial-y, frequency/time) to match the spatio-temporal RF tensor.

Input tensor layout  : (B, C_in,  D, H, W)
                         B  = batch
                         C  = channels (1 for raw power map)
                         D  = frequency / time slices
                         H,W = spatial grid
Binary mask layout   : (B,  1,    D, H, W)  –  1 = measured, 0 = missing
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class PartialConv3d(nn.Module):
    """
    3-D Partial Convolution layer.

    Parameters
    ----------
    in_channels  : int  – number of input feature channels
    out_channels : int  – number of output feature channels
    kernel_size  : int or 3-tuple
    stride       : int or 3-tuple
    padding      : int or 3-tuple
    bias         : bool – include a bias term (default True)
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        bias: bool = True,
    ) -> None:
        super().__init__()

        # Main convolution – applied to masked-zeroed input
        self.conv = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,          # bias handled manually after scaling
        )

        # Mask update kernel: all-ones, single-channel, no learning
        self.register_buffer(
            "mask_kernel",
            torch.ones(1, 1, *self._to_tuple(kernel_size)),
        )

        self.stride = self._to_tuple(stride)
        self.padding = self._to_tuple(padding)

        # Receptive field size (used for normalisation)
        k = self._to_tuple(kernel_size)
        self.rf_size = float(k[0] * k[1] * k[2]) * in_channels

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.bias = None

        self._init_weights()

    # ------------------------------------------------------------------
    # helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _to_tuple(v):
        return (v, v, v) if isinstance(v, int) else tuple(v)

    def _init_weights(self):
        nn.init.kaiming_normal_(self.conv.weight, mode="fan_out",
                                nonlinearity="relu")

    # ------------------------------------------------------------------
    # forward
    # ------------------------------------------------------------------
    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor,
    ):
        """
        Parameters
        ----------
        x    : (B, C_in, D, H, W)  – raw input (zeros at missing sites)
        mask : (B,  1,   D, H, W)  – binary mask (1=valid, 0=missing)

        Returns
        -------
        out       : (B, C_out, D', H', W')  – partial-conv activations
        mask_out  : (B,  1,    D', H', W')  – propagated binary mask
        """
        # ── Step 1: zero-out invalid pixels ──────────────────────────
        # mask is broadcast across channels
        x_masked = x * mask                      # (B, C, D, H, W)

        # ── Step 2: standard convolution on zeroed input ─────────────
        raw_out = self.conv(x_masked)            # (B, C_out, D', H', W')

        # ── Step 3: count valid pixels in each receptive field ───────
        # Use the all-ones mask kernel; sum gives |M ∩ RF|
        with torch.no_grad():
            # mask must be single-channel for this conv
            mask_sum = F.conv3d(
                mask,
                self.mask_kernel,
                stride=self.stride,
                padding=self.padding,
            )                                    # (B, 1, D', H', W')

        # ── Step 4: scale activations ────────────────────────────────
        # ratio = (total RF size) / (number of valid pixels)
        # Clamp denominator to avoid division by zero at fully-masked
        # locations (their output will later be zeroed by mask_out).
        scale = self.rf_size / (mask_sum + 1e-8)  # (B, 1, D', H', W')
        out = raw_out * scale                    # broadcast over C_out

        # ── Step 5: add bias ─────────────────────────────────────────
        if self.bias is not None:
            # bias shape: (C_out,) → (1, C_out, 1, 1, 1)
            out = out + self.bias.view(1, -1, 1, 1, 1)

        # ── Step 6: update mask ──────────────────────────────────────
        # Output location is valid iff mask_sum > 0
        with torch.no_grad():
            mask_out = (mask_sum > 0).float()   # (B, 1, D', H', W')

        # Zero activations at fully-masked output locations
        out = out * mask_out

        return out, mask_out


# ---------------------------------------------------------------------------
# Patch-Embedding stem built from stacked Partial Convolutions
# ---------------------------------------------------------------------------
class PartialConvStem(nn.Module):
    """
    A two-layer 3-D Partial Convolution stem that converts a raw sparse
    RF tensor into a dense feature map, while propagating the validity
    mask through each layer.

    Architecture
    ------------
    Input  (B, 1,        D,    H,    W)
    ↓  PartialConv3d(1  → stem_dim, k=3, s=1, p=1)
    ↓  BatchNorm3d + ReLU
    ↓  PartialConv3d(stem_dim → embed_dim, k=3, s=patch_stride, p=1)
    ↓  BatchNorm3d + ReLU
    Output (B, embed_dim, D//ps, H//ps, W//ps)

    Parameters
    ----------
    in_channels  : int – raw signal channels (typically 1)
    stem_dim     : int – intermediate channel width
    embed_dim    : int – final embedding dimension (= encoder hidden dim)
    patch_stride : int – spatial downsampling factor (patch size)
    """

    def __init__(
        self,
        in_channels: int = 1,
        stem_dim: int = 64,
        embed_dim: int = 256,
        patch_stride: int = 4,
    ) -> None:
        super().__init__()

        self.pconv1 = PartialConv3d(in_channels, stem_dim,
                                    kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(stem_dim)

        self.pconv2 = PartialConv3d(stem_dim, embed_dim,
                                    kernel_size=3,
                                    stride=patch_stride,
                                    padding=1)
        self.bn2 = nn.BatchNorm3d(embed_dim)

        self.act = nn.ReLU(inplace=True)
        self.patch_stride = patch_stride

    def forward(self, x: torch.Tensor, mask: torch.Tensor):
        """
        Parameters
        ----------
        x    : (B, C_in, D, H, W)
        mask : (B, 1,    D, H, W)

        Returns
        -------
        feat      : (B, embed_dim, D', H', W')
        mask_down : (B, 1,         D', H', W')
        """
        x, mask = self.pconv1(x, mask)
        x = self.act(self.bn1(x))

        x, mask = self.pconv2(x, mask)
        x = self.act(self.bn2(x))

        return x, mask
